In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
# from tensorflow_addons.losses import SigmoidFocalCrossEntropy
from sklearn.metrics import precision_recall_curve
# Load the dataset
# file_path = "updated_combined_data_with_actual_volume.xlsx"
file_path = "data_without_straw.xlsx"
data = pd.read_excel(file_path)
import random, numpy as np, tensorflow as tf
# data.sort_values(by=['Participant_ID', 'Time'], inplace=True)

# Preprocessing Data
# Convert time to a useful format if necessary and encode labels
data['Label'] = LabelEncoder().fit_transform(data['Label'])

# Assuming each 'Time' captures a snapshot of zones, we can reshape the data
X = data.iloc[:, 1:65].values  # Zone_0 to Zone_63
y = data['Label'].values

# Reshape X for CNN-LSTM, e.g., (samples, time_steps, height, width, channels)
time_steps = 5  # Number of time steps to consider (you can adjust this)
X_seq = []
y_seq = []

for i in range(len(X) - time_steps):
    X_seq.append(X[i:i + time_steps])  # Capture sequential data
    y_seq.append(y[i + time_steps])     # Corresponding label after the sequence ends
train_ratio = 0.8 

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)
# X_train, X_test, y_train, y_test = train_test_split(X_seq, y_seq, test_size=0.2, shuffle=False)

# y_train = to_categorical(y_train, num_classes=2)  # Assuming two classes for drinking
# y_test = to_categorical(y_test, num_classes=2)

In [2]:
import pandas as pd, numpy as np, random, tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------------
# 1)  Load + encode labels
# ------------------------------------------------------------------
file_path = "data_without_straw.xlsx"
data = pd.read_excel(file_path)

le = LabelEncoder()
data["Label"] = le.fit_transform(data["Label"])     # text → int

X = data.iloc[:, 1:65].values.astype("float32")     # zone features
y = data["Label"].values.astype("int8")             # frame labels

print("Label mapping:")
for i, name in enumerate(le.classes_):
    print(f"  {i} → {name}")

# ------------------------------------------------------------------
# 2)  Identify majority / minority *at frame level*
# ------------------------------------------------------------------
from collections import Counter
freq = Counter(y)          # e.g. {0: 18_000, 1: 3_000}
majority_id, minority_id = max(freq, key=freq.get), min(freq, key=freq.get)
print(f"\nMajority label  : {majority_id} ({le.inverse_transform([majority_id])[0]})")
print(f"Minority label  : {minority_id} ({le.inverse_transform([minority_id])[0]})")

# ------------------------------------------------------------------
# 3)  Window generation with class-dependent stride
# ------------------------------------------------------------------
time_steps     = 5          # window length
stride_major   = 17          # skip 4 frames inside majority runs
stride_minor   = 1          # slide by 1 inside minority runs

X_seq, y_seq = [], []
i = 0
while i + time_steps < len(X):
    lbl_next = y[i + time_steps]          # window label = frame at t + time_steps
    X_seq.append(X[i : i + time_steps])
    y_seq.append(lbl_next)

    # advance differently depending on which class we just added
    i += stride_major if lbl_next == majority_id else stride_minor

X_seq = np.asarray(X_seq, dtype="float32")
y_seq = np.asarray(y_seq,  dtype="int8")

# ------------------------------------------------------------------
# 4)  Show the new balance
# ------------------------------------------------------------------
seq_freq = Counter(y_seq)
total_seq = len(y_seq)
print("\nSequence-level distribution AFTER variable stride:")
for k in sorted(seq_freq):
    pct = seq_freq[k] / total_seq * 100
    print(f"  {le.inverse_transform([k])[0]:<15} {seq_freq[k]:>7}  ({pct:.1f} %)")

# ------------------------------------------------------------------
# 5)  Train / test split  +  one-hot labels
# ------------------------------------------------------------------
train_ratio = 0.90  # 90% training, 10% testing

# Get indices for each class
unique_labels, indices = {}, {}
for label in np.unique(y_seq):
    indices[label] = np.where(y_seq == label)[0]

# Create train and test lists
X_train_indices, y_train_indices = [], []
X_test_indices, y_test_indices = [], []

for label, idx in indices.items():
    n = len(idx)
    split_point = int(n * train_ratio)
    # Select training and testing indices while preserving order
    X_train_indices.extend(idx[:split_point])
    y_train_indices.extend([label] * split_point)
    X_test_indices.extend(idx[split_point:])
    y_test_indices.extend([label] * (n - split_point))

# Create train and test sets
X_train = X_seq[X_train_indices]
y_train = y_seq[y_train_indices]
X_test = X_seq[X_test_indices]
y_test = y_seq[y_test_indices]

# Convert ylabel to categorical
y_train = to_categorical(y_train, num_classes=len(le.classes_))
y_test = to_categorical(y_test, num_classes=len(le.classes_))

# Check shapes
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Label mapping:
  0 → Drinking
  1 → Not_Drinking

Majority label  : 1 (Not_Drinking)
Minority label  : 0 (Drinking)

Sequence-level distribution AFTER variable stride:
  Drinking           3035  (40.1 %)
  Not_Drinking       4532  (59.9 %)
Shape of X_train: (6809, 5, 64)
Shape of X_test: (758, 5, 64)
Shape of y_train: (6809, 2)
Shape of y_test: (758, 2)


In [3]:
y_true = np.argmax(y_test, axis=1) 

In [4]:
data['Label'].value_counts()

Label
1    74730
0     5345
Name: count, dtype: int64

In [5]:
X_train

array([[[2487.,  165., 2821., ...,  383.,  395., 1900.],
        [2487.,  165., 2821., ...,  394.,  382., 1901.],
        [2487.,  165., 2821., ...,  394.,  382., 1901.],
        [2487.,  165.,  348., ...,  364.,  330., 1817.],
        [2487.,  165.,  348., ...,  364.,  330., 1817.]],

       [[2487.,  165., 2821., ...,  394.,  382., 1901.],
        [2487.,  165., 2821., ...,  394.,  382., 1901.],
        [2487.,  165.,  348., ...,  364.,  330., 1817.],
        [2487.,  165.,  348., ...,  364.,  330., 1817.],
        [ 265.,  200.,  234., ...,  364.,  352., 1817.]],

       [[2487.,  165., 2821., ...,  394.,  382., 1901.],
        [2487.,  165.,  348., ...,  364.,  330., 1817.],
        [2487.,  165.,  348., ...,  364.,  330., 1817.],
        [ 265.,  200.,  234., ...,  364.,  352., 1817.],
        [ 178.,  200.,  215., ...,  338.,  352., 1783.]],

       ...,

       [[ 256.,  242.,  250., ...,  192.,  223.,  254.],
        [ 256.,  242.,  250., ...,  192.,  224.,  254.],
        [ 26

In [6]:
y_train


array([[0., 1.],
       [0., 1.],
       [0., 1.],
       ...,
       [0., 1.],
       [0., 1.],
       [0., 1.]])

In [7]:
np.count_nonzero(y_test == 1)

758

In [8]:
np.count_nonzero(y_train == [1, 0])

0

In [9]:
labels = np.argmax(y_test, axis=1)          # turns [1,0] -> 0; [0,1] -> 1
unique, counts = np.unique(labels, return_counts=True)
print(dict(zip(unique, counts)))

{np.int64(1): np.int64(758)}


In [10]:
labels = np.argmax(y_train, axis=1)          # turns [1,0] -> 0; [0,1] -> 1
unique, counts = np.unique(labels, return_counts=True)
print(dict(zip(unique, counts)))

{np.int64(1): np.int64(6809)}


In [11]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

def create_regularised_model(input_shape,
                             weight_decay=5e-5,
                             conv_dropout=0.3,
                             lstm_dropout=0.99):

    L2 = regularizers.l2(weight_decay)
    model = models.Sequential()
    model.add(layers.Input(shape=input_shape))               # (T, H, W, C)

    # --- Convolutional feature extractor ---
    for filters in (32, 64, 128):
        model.add(layers.TimeDistributed(
            layers.Conv2D(filters, (3, 3),
                          padding='same',
                          activation='relu',
                          kernel_regularizer=L2)))
        model.add(layers.TimeDistributed(layers.BatchNormalization()))
        model.add(layers.TimeDistributed(layers.MaxPooling2D()))
        # SpatialDropout2D zeroes entire feature maps ➜ stronger regularisation
        model.add(layers.TimeDistributed(layers.SpatialDropout2D(conv_dropout)))

    # Flatten the per-frame feature maps
    model.add(layers.TimeDistributed(layers.Flatten()))

    # --- Temporal modelling with Bi-LSTM ---
    model.add(layers.Bidirectional(
        layers.LSTM(128,
                    return_sequences=True,
                    dropout=lstm_dropout,          # input dropout
                    recurrent_dropout=lstm_dropout, # recurrent dropout
                    kernel_regularizer=L2)))

    # Dropout *between* the two LSTMs
    model.add(layers.Dropout(lstm_dropout))

    model.add(layers.Bidirectional(
        layers.LSTM(64,
                    dropout=lstm_dropout,
                    recurrent_dropout=lstm_dropout,
                    kernel_regularizer=L2)))

    # --- Classification head ---
    model.add(layers.Dense(64, activation='relu',
                           kernel_regularizer=L2))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(2, activation='softmax',
                           kernel_regularizer=L2))

    return model

# Create input shape based on the sequence length and pixel size
input_shape = (time_steps, 8, 8, 1)  # (timesteps, height, width, channels)
model = create_regularised_model(
            input_shape,
            weight_decay=7e-7,   # softer L2
            conv_dropout=0.3,   # weaker SpatialDropout2D
            lstm_dropout=0.10) 
# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Model summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 5, 8, 8, 32)    │           320 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 5, 8, 8, 32)    │           128 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 5, 4, 4, 32)    │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 5, 4, 4, 32)    │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_4              │ (None, 5, 4, 4, 64)    │        18,496 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_5              │ (None, 5, 4, 4, 64)    │           256 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_6              │ (None, 5, 2, 2, 64)    │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_7              │ (None, 5, 2, 2, 64)    │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_8              │ (None, 5, 2, 2, 128)   │        73,856 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_9              │ (None, 5, 2, 2, 128)   │           512 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_10             │ (None, 5, 1, 1, 128)   │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_11             │ (None, 5, 1, 1, 128)   │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_12             │ (None, 5, 128)         │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 5, 256)         │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 5, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 529,474 (2.02 MB)

 Trainable params: 529,026 (2.02 MB)

 Non-trainable params: 448 (1.75 KB)

In [12]:
# ================================================================
#  Cross-validation + pick-the-best + full test-set evaluation
# ================================================================
import numpy as np
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import class_weight
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold

n_splits   = 5
best_acc   = -np.inf
best_wgt   = None               # to store best-performing weights
acc_per_fold, loss_per_fold = [], []

skf = StratifiedKFold(n_splits=n_splits, shuffle=False)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_seq, y_seq), 1):
    print(f'\n――――――  Fold {fold}/{n_splits}  ――――――')

    # -------- Split & reshape -------------------------------------
    X_train, X_val = X_seq[train_idx], X_seq[val_idx]
    y_train, y_val = y_seq[train_idx], y_seq[val_idx]

    X_train = X_train.reshape(-1, time_steps, 8, 8, 1)
    X_val   = X_val.reshape( -1, time_steps, 8, 8, 1)

    y_train = to_categorical(y_train, num_classes=2)
    y_val   = to_categorical(y_val,   num_classes=2)

    # -------- Fresh model per fold --------------------------------
    model = create_regularised_model(
        input_shape=(time_steps, 8, 8, 1),
        weight_decay=7e-7,
        conv_dropout=0.3,
        lstm_dropout=0.10
    )
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    # -------- Balanced class weights ------------------------------
    cw = class_weight.compute_class_weight(
            class_weight='balanced',
            classes=np.unique(y_seq[train_idx]),
            y=y_seq[train_idx])
    cw = dict(enumerate(cw))

    # -------- Callbacks -------------------------------------------
    es = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=3,
            restore_best_weights=True, verbose=0)

    # -------- Train ----------------------------------------------
    model.fit(X_train, y_train,
              epochs=50, batch_size=32,
              validation_data=(X_val, y_val),
              class_weight=cw,
              callbacks=[es],
              verbose=0)

    # -------- Evaluate -------------------------------------------
    loss, acc = model.evaluate(X_val, y_val, verbose=0)
    print(f'Fold {fold}:  val_loss = {loss:.4f}   val_acc = {acc:.4f}')

    acc_per_fold.append(acc)
    loss_per_fold.append(loss)

    if acc > best_acc:
        best_acc = acc
        best_wgt = model.get_weights()
        print(f'   ↳ New best model found (acc = {best_acc:.4f})')

# ================================================================
#  Cross-validation summary
# ================================================================
print('\n>>>>  CV results  <<<<')
print(f'Accuracy : {np.mean(acc_per_fold):.4f} ± {np.std(acc_per_fold):.4f}')
print(f'Loss     : {np.mean(loss_per_fold):.4f} ± {np.std(loss_per_fold):.4f}')

# ================================================================
#  Reload the best fold and test on held-out data
# ================================================================
best_model = create_regularised_model(
    input_shape=(time_steps, 8, 8, 1),
    weight_decay=7e-7,
    conv_dropout=0.3,
    lstm_dropout=0.10
)
best_model.compile(optimizer='adam',
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])
best_model.set_weights(best_wgt)            # <- load winning weights

# Prepare test-set tensors
X_test = X_test.reshape(-1, time_steps, 8, 8, 1)
y_test = np.asarray(y_test).squeeze()
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=2)
print("X_test shape:", X_test.shape)
print("y_test shape before:", y_test.shape)
if len(y_test.shape) > 1 and y_test.shape[1] == 2:
    # Already one-hot!
    y_test_cat = y_test
else:
    # Not one-hot, must convert
    y_test = np.asarray(y_test).squeeze()
    y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=2)
print("y_test shape after:", y_test.shape)
print("y_test_cat shape after:", y_test_cat.shape)

# --- Raw scores ---------------------------------------------------
test_loss, test_acc = best_model.evaluate(X_test, y_test_cat, verbose=0)
print(f'\n>>>>  Test-set performance  <<<<')
print(f'Loss: {test_loss:.4f}   Accuracy: {test_acc:.4f}')

# --- Predict class labels ----------------------------------------
prob = best_model.predict(X_test, verbose=0)
y_pred = np.argmax(prob, axis=1)
y_true = y_test

# --- Classic metrics ---------------------------------------------
prec = precision_score(y_true, y_pred)
rec  = recall_score(y_true, y_pred)
f1   = f1_score(y_true, y_pred)
auc  = roc_auc_score(y_true, prob[:,1])

print('\nDetailed metrics:')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1-score  : {f1:.4f}')
print(f'ROC-AUC   : {auc:.4f}')

# ================================================================
#  Confusion matrix
# ================================================================
cm = confusion_matrix(y_true, y_pred)
labels = ['Class 0', 'Class 1']   # change names if you have better labels

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix – Best CV Model on Test Set')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()


――――――  Fold 1/5  ――――――
Fold 1:  val_loss = 0.8269   val_acc = 0.7232
   ↳ New best model found (acc = 0.7232)

――――――  Fold 2/5  ――――――
Fold 2:  val_loss = 0.4708   val_acc = 0.8316
   ↳ New best model found (acc = 0.8316)

――――――  Fold 3/5  ――――――
Fold 3:  val_loss = 0.3870   val_acc = 0.8619
   ↳ New best model found (acc = 0.8619)

――――――  Fold 4/5  ――――――
Fold 4:  val_loss = 0.5639   val_acc = 0.7317

――――――  Fold 5/5  ――――――
Fold 5:  val_loss = 0.4771   val_acc = 0.8030

>>>>  CV results  <<<<
Accuracy : 0.7903 ± 0.0546
Loss     : 0.5451 ± 0.1516
X_test shape: (758, 5, 8, 8, 1)
y_test shape before: (758, 2)
y_test shape after: (758, 2)
y_test_cat shape after: (758, 2)

>>>>  Test-set performance  <<<<
Loss: 1.4101   Accuracy: 0.5383


ValueError: Classification metrics can't handle a mix of multilabel-indicator and binary targets

In [ ]:
# --- Predict class labels and prepare y_true as integers ---------
prob = best_model.predict(X_test, verbose=0)
y_pred = np.argmax(prob, axis=1)
best_model.save('Best_gesture_recognition_model.h5')
# y_test_cat is likely (N, 2) one-hot, so convert back to integer class labels
if len(y_test_cat.shape) > 1 and y_test_cat.shape[1] == 2:
    y_true_labels = np.argmax(y_test_cat, axis=1)
else:
    y_true_labels = y_test_cat

# --- Classic metrics ---------------------------------------------
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

prec = precision_score(y_true_labels, y_pred)
rec  = recall_score(y_true_labels, y_pred)
f1   = f1_score(y_true_labels, y_pred)
auc  = roc_auc_score(y_true_labels, prob[:,1])

print('\nDetailed metrics:')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1-score  : {f1:.4f}')
print(f'ROC-AUC   : {auc:.4f}')

# --- Confusion matrix --------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_true_labels, y_pred)
labels = ['Class 0', 'Class 1']   # change this list if you have more descriptive names

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix – Best CV Model on Test Set')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
# ----- Prepare test-set tensors -----
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

y_test = np.asarray(y_test).squeeze()
print("y_test shape after squeeze:", y_test.shape)

if len(y_test.shape) > 1 and y_test.shape[1] == 2:
    # Already one-hot!
    y_test_cat = y_test
    y_true = np.argmax(y_test, axis=1)
else:
    # Not one-hot, must convert
    y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=2)
    y_true = y_test  # already int labels
print("y_test_cat shape (final):", y_test_cat.shape)

# --- Raw scores ---------------------------------------------------
test_loss, test_acc = best_model.evaluate(X_test, y_test_cat, verbose=0)
print(f'\n>>>>  Test-set performance  <<<<')
print(f'Loss: {test_loss:.4f}   Accuracy: {test_acc:.4f}')

# --- Predict class labels ----------------------------------------
prob = best_model.predict(X_test, verbose=0)
y_pred = np.argmax(prob, axis=1)

# --- Classic metrics ---------------------------------------------
prec = precision_score(y_true, y_pred)
rec  = recall_score(y_true, y_pred)
f1   = f1_score(y_true, y_pred)
auc  = roc_auc_score(y_true, prob[:,1])

print('\nDetailed metrics:')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1-score  : {f1:.4f}')
print(f'ROC-AUC   : {auc:.4f}')

In [ ]:
# Confusion matrix
conf_matrix = confusion_matrix(y_true_classes, y_pred_classes)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=["Drinking", "Not Drinking"], yticklabels=["Drinking", "Not Drinking"])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# If you have a regression model for Actual Volume, replace the following
# with y_true_volume and y_pred_volume definitions as appropriate.

# Assuming you have y_true_volume and y_pred_volume
from sklearn.metrics import mean_squared_error

# Replace with actual volume predictions when regression model is created
# y_pred_volume will typically be obtained from your regression model
# y_true_volume should be the true values from the dataset
y_true_volume = data['Actual_Volume'].values  # Use your actual volumes for regression

# Predict volumes using your regression model (e.g., predict_model) if it's separate
# This is just a placeholder for where to put your actual volume predictions
# y_pred_volume = regression_model.predict(X_volume)

# Example MSE and RMSE calculation
mse = mean_squared_error(y_true_volume, y_pred_volume)
rmse = np.sqrt(mse)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

In [ ]:
import numpy as np
import pandas as pd

# ---- 1.  Frame-level balance -----------------------------------
frame_counts = pd.Series(y).value_counts().sort_index()
print("Frame-level labels")
print(frame_counts.rename(index={0: "Not-Drinking", 1: "Drinking"}))
print()

# ---- 2.  Sequence-level balance (after the sliding window) -----
seq_counts = pd.Series(y_seq).value_counts().sort_index()
print("Sequence-level labels (y_seq)")
print(seq_counts.rename(index={0: "Not-Drinking", 1: "Drinking"}))

# ---- 3.  Quick percentages -------------------------------------
pct = (seq_counts / seq_counts.sum() * 100).round(1)
print(f"\nSequence-level ratio: {pct[0]} % Not-Drinking  /  {pct[1]} % Drinking")